In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import tensorflow as tf
import json
import datetime
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
import os

warnings.filterwarnings('ignore')
tf.get_logger().setLevel('ERROR')

# ========================= CONFIG =========================
IMG_SIZE = (224, 224)
BATCH_SIZE = 12
SEED = 123
VALIDATION_SPLIT = 0.2

# ========================= PATHS =========================
DATA_DIR = Path("/kaggle/input/datasets/aayushraman07/plant-dataset/plantvillage dataset/segmented")
MODELS_DIR = Path("/kaggle/working/models")
LOGS_DIR = Path("/kaggle/working/logs")

MODELS_DIR.mkdir(exist_ok=True)
LOGS_DIR.mkdir(exist_ok=True)

print(f"Using Data Directory: {DATA_DIR}")

# ========================= PERFORMANCE OPTIMIZATIONS =========================
print("Applying Performance Optimizations...")

# CPU Optimizations
tf.config.threading.set_intra_op_parallelism_threads(4)
tf.config.threading.set_inter_op_parallelism_threads(2)
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '1'

# GPU Config
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"{len(gpus)} GPU(s) detected")
    try:
        
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU Memory Growth enabled successfully")
    except RuntimeError as e:
        print(" Memory growth setting failed (normal if GPUs already initialized):", e)
    
    tf.keras.mixed_precision.set_global_policy('mixed_float16')
    print(" Mixed Precision enabled")
else:
    print("No GPU detected! Running on CPU.")

# ========================= DATA LOADING =========================
def load_datasets(data_dir):
    train_ds = tf.keras.preprocessing.image_dataset_from_directory(
        data_dir,
        validation_split=VALIDATION_SPLIT,
        subset="training",
        seed=SEED,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=True
    )
    val_ds = tf.keras.preprocessing.image_dataset_from_directory(
        data_dir,
        validation_split=VALIDATION_SPLIT,
        subset="validation",
        seed=SEED,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE
    )
    return train_ds, val_ds, train_ds.class_names


# ========================= PLOTTING =========================
def plot_training_history(history, title="Training History", save_path=None):
    if not history:
        return
    acc = history.history['accuracy']
    val_acc = history.history['val_accuracy']
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    epochs = range(1, len(acc) + 1)

    plt.figure(figsize=(14, 5))
    plt.subplot(1, 2, 1)
    plt.plot(epochs, acc, 'b-', label='Training Accuracy')
    plt.plot(epochs, val_acc, 'r-', label='Validation Accuracy')
    plt.title(f'{title} - Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)

    plt.subplot(1, 2, 2)
    plt.plot(epochs, loss, 'b-', label='Training Loss')
    plt.plot(epochs, val_loss, 'r-', label='Validation Loss')
    plt.title(f'{title} - Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Plot saved: {save_path}")
    plt.show()


# ========================= MAIN =========================
if __name__ == "__main__":
    print(" Starting Plant Disease Classification Training...\n")

    train_data, val_data, class_names = load_datasets(DATA_DIR)
    num_classes = len(class_names)
    print(f"Found {num_classes} classes")
    print("Sample classes:", class_names[:8])

    with open("/kaggle/working/class_names.json", "w") as f:
        json.dump(class_names, f, indent=2)

    # ===============Data Pipeline=============================
    AUTOTUNE = tf.data.AUTOTUNE
    train_data = train_data.shuffle(1000).prefetch(AUTOTUNE)
    val_data = val_data.cache("/tmp/val_cache").prefetch(AUTOTUNE)

    # Callbacks
    callbacks = [
        tf.keras.callbacks.ModelCheckpoint("/kaggle/working/plant_model_best.keras",
                                         monitor="val_accuracy", save_best_only=True, verbose=1),
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, verbose=1),
        tf.keras.callbacks.TensorBoard(log_dir=str(LOGS_DIR / datetime.datetime.now().strftime("%Y%m%d-%H%M%S")))
    ]

    # Model
    data_augmentation = tf.keras.Sequential([
        tf.keras.layers.RandomFlip("horizontal_and_vertical"),
        tf.keras.layers.RandomRotation(0.2),
        tf.keras.layers.RandomZoom(0.25),
        tf.keras.layers.RandomContrast(0.2),
    ])

    base_model = tf.keras.applications.MobileNetV2(   
        input_shape=(*IMG_SIZE, 3),
        include_top=False,
        weights='imagenet',
        alpha=0.75
    )
    base_model.trainable = False

    model = tf.keras.Sequential([
        tf.keras.Input(shape=(*IMG_SIZE, 3)),
        data_augmentation,
        tf.keras.layers.Lambda(tf.keras.applications.mobilenet_v2.preprocess_input),
        base_model,
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.4),
        tf.keras.layers.Dense(num_classes, activation='softmax', dtype='float32')
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    print("\nPhase 1: Training top layers...")
    history1 = model.fit(train_data, validation_data=val_data, epochs=10, callbacks=callbacks)
    plot_training_history(history1, "Phase 1 - Feature Extraction", "/kaggle/working/phase1_training.png")

    print("\nPhase 2: Fine-tuning...")
    base_model.trainable = True
    for layer in base_model.layers:
        layer.trainable = any(x in layer.name for x in ["block_13", "block_14", "block_15"])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    history2 = model.fit(train_data, validation_data=val_data, epochs=20, callbacks=callbacks)
    plot_training_history(history2, "Phase 2 - Fine Tuning", "/kaggle/working/phase2_training.png")

    print("\nTraining Completed!")
    model.save("/kaggle/working/plant_disease_model_final.keras")
    model.summary()
    plot_training_history(history2, "Final Training Results", "/kaggle/working/final_training_history.png")


In [ ]:
import tensorflow as tf
import json
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

# ========================= CONFIG =========================
IMG_SIZE = (224, 224)
MODEL_PATH = "/kaggle/working/plant_disease_model_final.keras"
CLASS_NAMES_PATH = "/kaggle/working/class_names.json"

# ========================= LOAD MODEL & CLASSES =========================
print("Loading model and class names...")

model = tf.keras.models.load_model(MODEL_PATH)
print("Model loaded successfully!")

with open(CLASS_NAMES_PATH, "r") as f:
    class_names = json.load(f)

print(f"Loaded {len(class_names)} classes")

# ========================= PREDICTION FUNCTION =========================
def predict_disease(image_path):
    """Predict disease from a single image"""
    # Load image
    img = tf.keras.preprocessing.image.load_img(image_path, target_size=IMG_SIZE)
    img_array = tf.keras.preprocessing.image.img_to_array(img)
    img_array = tf.expand_dims(img_array, 0)
    
   
    
    predictions = model.predict(img_array, verbose=0)
    predicted_idx = np.argmax(predictions[0])
    confidence = float(predictions[0][predicted_idx] * 100)
    predicted_class = class_names[predicted_idx]
    
    print(f"\n **Prediction** : {predicted_class}")
    print(f" **Confidence** : {confidence:.2f}%")
    
    
    plt.figure(figsize=(6, 6))
    plt.imshow(img)
    plt.title(f"{predicted_class} ({confidence:.1f}%)")
    plt.axis('off')
    plt.show()
    
    return predicted_class, confidence


# ========================= TEST PREDICTION =========================
if __name__ == "__main__":

    test_image_path = "/kaggle/input/datasets/aayushraman07/test-leaf/test_leaf.jpg"
    
    if Path(test_image_path).exists():
        predict_disease(test_image_path)
    else:
        print(f"Image not found: {test_image_path}")
        print("Please update the test_image_path")